# Extra: detección de matrículas con YOLO frente a RT-DETR

Este cuaderno compara dos detectores de matrículas entrenados con **el mismo dataset** (`dataset_matriculas`, mismo reparto train / val / test):

| | YOLO11n | RT-DETR-L |
|---|---|---|
| Arquitectura | CNN, una etapa, con NMS | *Transformer* (DETR en tiempo real), sin NMS |
| Licencia | AGPL-3.0 | Apache 2.0 (modelo original de Baidu) |
| Entrenamiento | `VC_P4.ipynb` (ya hecho) | este cuaderno |

Se usa la implementación de RT-DETR incluida en Ultralytics (`from ultralytics import RTDETR`). Así el entrenamiento y la evaluación se hacen con el mismo `.yaml`, las mismas métricas y el mismo código que en YOLO, y la comparación es justa.

Pasos:
1. Entrenar RT-DETR-L para la clase `matricula`.
2. Analizar el entrenamiento (pérdidas y métricas) y compararlo con el de YOLO.
3. Evaluar los dos modelos en el **conjunto de test**: precisión, *recall*, mAP, velocidad y tamaño.
4. Comparar visualmente sobre imágenes de test.
5. Comparar sobre el vídeo de test, buscando la matrícula dentro de los vehículos detectados, igual que en el prototipo.

> **Tiempo de entrenamiento:** RT-DETR-L tiene unos 32 M de parámetros, frente a los 2,6 M de YOLO11n.
> En CPU, 40 épocas pueden llevar **más de un día**. Con GPU (por ejemplo, en Google Colab) son un par de horas.

### Entrenamiento en Google Colab

En CPU, el entrenamiento de RT-DETR-L tardaría días. Por eso se entrenó en Google Colab (GPU Tesla T4) con el cuaderno
[`VC_P4_RTDETR_colab.ipynb`](VC_P4_RTDETR_colab.ipynb), que clona este repositorio, entrena con los mismos ajustes que YOLO
y guarda el progreso en Google Drive. Tardó **77 minutos** en completar las 40 épocas.

Los resultados se copiaron en `runs/detect/matriculas_rtdetr_l`. Este cuaderno detecta que los pesos ya existen,
no vuelve a entrenar y hace el análisis y la evaluación en local (CPU).

## 1. Configuración

In [ ]:
import os, glob, random, time
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO, RTDETR

DATA_DIR    = os.path.abspath('dataset_matriculas')
RUNS_DIR    = os.path.abspath(os.path.join('runs', 'detect'))
OUT_DIR     = 'resultados'
os.makedirs(OUT_DIR, exist_ok=True)

YOLO_DIR    = os.path.join(RUNS_DIR, 'matriculas_yolo11n')
YOLO_PT     = os.path.join(YOLO_DIR, 'weights', 'best.pt')
RTDETR_NAME = 'matriculas_rtdetr_l'
RTDETR_DIR  = os.path.join(RUNS_DIR, RTDETR_NAME)
RTDETR_PT   = os.path.join(RTDETR_DIR, 'weights', 'best.pt')
RTDETR_LAST = os.path.join(RTDETR_DIR, 'weights', 'last.pt')

assert os.path.isdir(os.path.join(DATA_DIR, 'train', 'images')), 'No se encuentra dataset_matriculas (ejecuta antes VC_P4.ipynb)'
assert os.path.exists(YOLO_PT), 'No se encuentran los pesos de YOLO (runs/detect/matriculas_yolo11n)'

# Este .yaml usa la ruta absoluta del equipo actual, así que funciona tanto en local como en Colab.
# Es el mismo reparto que usó YOLO.
YAML_PATH = os.path.join(DATA_DIR, 'matriculas_rtdetr.yaml')
with open(YAML_PATH, 'w') as f:
    f.write(f"path: {DATA_DIR.replace(os.sep, '/')}\n"
            "train: train/images\nval: val/images\ntest: test/images\n\n"
            "nc: 1\nnames: ['matricula']\n")

GPU    = torch.cuda.is_available()
DEVICE = 0 if GPU else 'cpu'
BATCH  = 16 if GPU else 4          # RT-DETR-L ocupa mucha memoria; en CPU conviene un batch pequeño
print('Dispositivo:', torch.cuda.get_device_name(0) if GPU else 'CPU', '| batch:', BATCH)
for split in ['train', 'val', 'test']:
    print(f"{split:5s}: {len(glob.glob(os.path.join(DATA_DIR, split, 'images', '*')))} imágenes")

## 2. Entrenamiento de RT-DETR-L

Se parte de `rtdetr-l.pt` (preentrenado en COCO). Se usan los mismos ajustes que en YOLO (`imgsz=640`, 40 épocas, `patience=10`) para que la comparación sea justa.

- Si ya existen los pesos `best.pt`, la celda no vuelve a entrenar.
- Si el entrenamiento se corta (por ejemplo, porque Colab cierra la sesión), pon `REANUDAR = True` para continuar desde `last.pt`.

In [ ]:
EPOCHS   = 40
REANUDAR = False

if REANUDAR and os.path.exists(RTDETR_LAST):
    RTDETR(RTDETR_LAST).train(resume=True)
elif not os.path.exists(RTDETR_PT):
    modelo = RTDETR('rtdetr-l.pt')
    modelo.train(data=YAML_PATH, epochs=EPOCHS, imgsz=640, batch=BATCH, device=DEVICE,
                 workers=8, patience=10, project=RUNS_DIR, name=RTDETR_NAME, exist_ok=True,
                 seed=0, deterministic=False, plots=True)
else:
    print('Pesos ya entrenados:', RTDETR_PT)

## 3. Análisis del entrenamiento

Las pérdidas de RT-DETR no son las mismas que las de YOLO:

- **giou_loss:** solapamiento de las cajas.
- **cls_loss:** clasificación.
- **l1_loss:** distancia L1 entre las coordenadas de las cajas.

Por eso se muestran por separado. Las **métricas de validación**, en cambio, sí son directamente comparables entre los dos modelos.

In [ ]:
def lee_resultados(carpeta):
    r = pd.read_csv(os.path.join(carpeta, 'results.csv'))
    r.columns = [c.strip() for c in r.columns]
    return r

res_rt = lee_resultados(RTDETR_DIR)
res_yo = lee_resultados(YOLO_DIR)

# Pérdidas de RT-DETR (entrenamiento y validación)
perdidas = [c.split('/')[1] for c in res_rt.columns if c.startswith('train/') and c.endswith('loss')]
fig, axs = plt.subplots(1, len(perdidas), figsize=(6 * len(perdidas), 4.5))
for ax, p in zip(np.atleast_1d(axs), perdidas):
    ax.plot(res_rt['epoch'], res_rt[f'train/{p}'], label='train')
    if f'val/{p}' in res_rt:
        ax.plot(res_rt['epoch'], res_rt[f'val/{p}'], label='val')
    ax.set_title(f'RT-DETR: {p}'); ax.set_xlabel('época'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

# Métricas de validación de los dos modelos, época a época
metricas = [('metrics/precision(B)', 'Precisión'), ('metrics/recall(B)', 'Recall'),
            ('metrics/mAP50(B)', 'mAP@0.5'), ('metrics/mAP50-95(B)', 'mAP@0.5:0.95')]
fig, axs = plt.subplots(1, 4, figsize=(22, 4.5))
for ax, (m, titulo) in zip(axs, metricas):
    ax.plot(res_yo['epoch'], res_yo[m], label='YOLO11n')
    ax.plot(res_rt['epoch'], res_rt[m], label='RT-DETR-L')
    ax.set_title(titulo + ' (validación)'); ax.set_xlabel('época'); ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

for nombre, r in [('YOLO11n', res_yo), ('RT-DETR-L', res_rt)]:
    b = r.loc[r['metrics/mAP50-95(B)'].idxmax()]
    print(f"{nombre:10s} épocas={len(r):3d}  mejor={int(b['epoch']):3d}  P={b['metrics/precision(B)']:.3f}  "
          f"R={b['metrics/recall(B)']:.3f}  mAP50={b['metrics/mAP50(B)']:.3f}  mAP50-95={b['metrics/mAP50-95(B)']:.3f}  "
          f"tiempo={r['time'].iloc[-1] / 3600:.1f} h")

In [ ]:
# Curvas PR y matrices de confusión de RT-DETR generadas por Ultralytics
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
for ax, f in zip(axs, ['BoxPR_curve.png', 'confusion_matrix_normalized.png']):
    p = os.path.join(RTDETR_DIR, f)
    if os.path.exists(p):
        ax.imshow(plt.imread(p))
    ax.set_title('RT-DETR: ' + f); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Evaluación en el conjunto de test

Los dos modelos se evalúan sobre las mismas 477 imágenes de test, que ninguno de ellos ha visto al entrenar, y en el mismo dispositivo.
La velocidad es el tiempo de inferencia por imagen que mide Ultralytics.

In [ ]:
modelos = {'YOLO11n': (YOLO, YOLO_PT), 'RT-DETR-L': (RTDETR, RTDETR_PT)}
filas = []
for nombre, (Clase, pt) in modelos.items():
    m = Clase(pt)
    n_param = sum(p.numel() for p in m.model.parameters()) / 1e6
    met = m.val(data=YAML_PATH, split='test', imgsz=640, batch=BATCH, device=DEVICE, plots=False, verbose=False,
                project=RUNS_DIR, name=f'test_{nombre}', exist_ok=True)
    filas.append({'modelo': nombre, 'parámetros (M)': n_param, 'pesos (MB)': os.path.getsize(pt) / 1e6,
                  'precisión': met.box.mp, 'recall': met.box.mr, 'mAP@0.5': met.box.map50,
                  'mAP@0.5:0.95': met.box.map, 'inferencia (ms/img)': met.speed['inference']})

comparacion = pd.DataFrame(filas).set_index('modelo').round(3)
comparacion.to_csv(os.path.join(OUT_DIR, 'comparacion_yolo_rtdetr.csv'))
display(comparacion)

In [ ]:
# Gráfico de barras con las métricas de test
cols = ['precisión', 'recall', 'mAP@0.5', 'mAP@0.5:0.95']
ax = comparacion[cols].T.plot.bar(figsize=(9, 4.5), rot=0)
ax.set_ylim(0, 1); ax.set_title('Detección de matrículas en test'); ax.grid(axis='y', alpha=.3)
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f', fontsize=8)
plt.tight_layout(); plt.show()

## 5. Comparación visual en imágenes de test

Fila superior: YOLO11n. Fila inferior: RT-DETR-L. En el título aparece el número de matrículas anotadas (GT) y el de detectadas.

In [ ]:
tests = sorted(glob.glob(os.path.join(DATA_DIR, 'test', 'images', '*.jpg')))
random.seed(7)
muestra = random.sample(tests, 4)
cargados = {n: C(pt) for n, (C, pt) in modelos.items()}

fig, axs = plt.subplots(2, 4, figsize=(20, 9))
for j, ip in enumerate(muestra):
    lp = ip.replace(os.sep + 'images' + os.sep, os.sep + 'labels' + os.sep).rsplit('.', 1)[0] + '.txt'
    n_gt = sum(1 for l in open(lp) if l.strip()) if os.path.exists(lp) else 0
    for i, (nombre, m) in enumerate(cargados.items()):
        r = m(ip, imgsz=640, conf=0.3, device=DEVICE, verbose=False)[0]
        axs[i, j].imshow(r.plot()[:, :, ::-1]); axs[i, j].axis('off')
        axs[i, j].set_title(f'{nombre}: {len(r.boxes)} det. / {n_gt} GT', fontsize=10)
plt.tight_layout(); plt.show()

## 6. Comparación sobre el vídeo de test

Se reproduce el uso real del prototipo:

1. `yolo11n.pt` detecta los vehículos con motor.
2. Se recorta cada vehículo.
3. Cada detector de matrículas busca la matrícula en ese recorte.

Para ahorrar tiempo se analiza 1 de cada `PASO` fotogramas.

El vídeo no tiene anotaciones de matrículas, así que aquí no se puede calcular precisión ni *recall*. Lo que se compara es:

- el **porcentaje de recortes en los que cada modelo encuentra matrícula**;
- la **confianza media** de esas detecciones;
- el **tiempo por recorte**.

También se muestran ejemplos en los que los modelos **no coinciden**.

In [ ]:
VIDEO     = 'C0142.MP4'
PASO      = 10
CONF      = 0.30
VEHICULOS = [2, 3, 5, 7]          # car, motorcycle, bus, truck (COCO)

if not os.path.exists(VIDEO):
    print(f'No se encuentra {VIDEO}; se omite la comparación en vídeo.')
else:
    det_veh = YOLO('yolo11n.pt')
    stats = {n: {'con_matricula': 0, 'conf': [], 'tiempo': 0.0} for n in cargados}
    discrepancias, n_recortes, n_frames = [], 0, 0

    cap = cv2.VideoCapture(VIDEO)
    f = 0
    while True:
        if f % PASO:
            if not cap.grab():
                break
            f += 1; continue
        ok, frame = cap.read()
        if not ok:
            break
        f += 1; n_frames += 1
        H, W = frame.shape[:2]

        recortes = []
        for b in det_veh(frame, classes=VEHICULOS, conf=CONF, device=DEVICE, verbose=False)[0].boxes:
            x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
            mx, my = int(0.05 * (x2 - x1)), int(0.05 * (y2 - y1))
            cx1, cy1, cx2, cy2 = max(0, x1 - mx), max(0, y1 - my), min(W, x2 + mx), min(H, y2 + my)
            if cx2 - cx1 > 16 and cy2 - cy1 > 16:
                recortes.append(frame[cy1:cy2, cx1:cx2])
        if not recortes:
            continue
        n_recortes += len(recortes)

        salida = {}
        for nombre, m in cargados.items():
            t0 = time.perf_counter()
            salida[nombre] = m(recortes, imgsz=640, conf=CONF, device=DEVICE, verbose=False)
            stats[nombre]['tiempo'] += time.perf_counter() - t0
            for r in salida[nombre]:
                if len(r.boxes):
                    stats[nombre]['con_matricula'] += 1
                    stats[nombre]['conf'].append(float(r.boxes.conf.max()))

        # Guardar algunos recortes donde solo uno de los dos modelos encuentra matrícula
        for k in range(len(recortes)):
            encontrado = {n: len(salida[n][k].boxes) > 0 for n in cargados}
            if len(set(encontrado.values())) > 1 and len(discrepancias) < 8:
                discrepancias.append({n: salida[n][k].plot() for n in cargados})
    cap.release()

    video_cmp = pd.DataFrame([{
        'modelo': n,
        'recortes con matrícula': s['con_matricula'],
        '% recortes': 100 * s['con_matricula'] / max(1, n_recortes),
        'confianza media': np.mean(s['conf']) if s['conf'] else np.nan,
        'ms por recorte': 1000 * s['tiempo'] / max(1, n_recortes),
    } for n, s in stats.items()]).set_index('modelo').round(3)
    print(f'{n_frames} fotogramas analizados (1 de cada {PASO}), {n_recortes} recortes de vehículos')
    video_cmp.to_csv(os.path.join(OUT_DIR, 'comparacion_video_yolo_rtdetr.csv'))
    display(video_cmp)

In [ ]:
# Recortes en los que los modelos no coinciden (columna izquierda: YOLO11n, derecha: RT-DETR-L)
if os.path.exists(VIDEO) and discrepancias:
    n = len(discrepancias)
    fig, axs = plt.subplots(n, 2, figsize=(8, 3 * n), squeeze=False)
    for i, d in enumerate(discrepancias):
        for j, nombre in enumerate(cargados):
            axs[i, j].imshow(d[nombre][:, :, ::-1]); axs[i, j].axis('off')
            if i == 0:
                axs[i, j].set_title(nombre)
    plt.tight_layout(); plt.show()
else:
    print('Sin discrepancias que mostrar.')

## 7. Conclusiones

_(Completar tras ejecutar el cuaderno con los valores obtenidos.)_

- **Precisión y *recall* en test:** ¿cuál localiza más matrículas (*recall*) y cuál se equivoca menos (precisión)? Hay que tener en cuenta que el *recall* de YOLO11n era el punto débil (≈0.60).
- **Calidad de las cajas (mAP@0.5:0.95):** ¿cuál ajusta mejor el contenedor a la matrícula?
- **Coste:** tamaño del modelo, tiempo de entrenamiento y tiempo de inferencia por imagen o por recorte. ¿Compensa la mejora, si la hay?
- **Vídeo:** ¿cuál encuentra matrícula en más vehículos? ¿Qué muestran los casos en los que no coinciden (matrículas lejanas, inclinadas, falsos positivos)?
- **Licencia:** RT-DETR (Apache 2.0) permite un uso comercial sin publicar el código, a diferencia de YOLO de Ultralytics (AGPL-3.0).
- **Elección final** para el prototipo y por qué.